In [1]:
import pandas as pd

df = pd.read_csv("source.csv")
df = df.drop(df.index[-1:])

In [2]:

df["delta"] = (df["close"] - df["open"]) / df["open"]

In [3]:
import numpy as np

statLength = 100

def _rolling_zscore_last(x):
    std = np.std(x)
    if std == 0 or np.isnan(std):
        return np.nan
    return (x[-1] - np.mean(x)) / std

df.loc[:, "price_effect"] = (df["high"] - df["low"]) * 0.25 + abs(df["close"] - df["open"]) * 0.75
df.loc[:, "vol_min"] = df["volume"].rolling(window=statLength, min_periods=statLength).min()
# avoid division by zero
df.loc[:, "vol_min"] = df.loc[:, "vol_min"].replace(0, np.nan)
df.loc[:, "vol_norm"] = df["volume"] / df.loc[:, "vol_min"]
df.loc[:, "pricePerVol"] = df.loc[:, "price_effect"] / df.loc[:, "vol_norm"]
# compute z-score within a rolling window and keep the z-score of the last element in each window
df.loc[:, "sentiment_zs"] = df["pricePerVol"].rolling(window=statLength, min_periods=statLength).apply(lambda x: _rolling_zscore_last(x), raw=True)


In [4]:

col = ["volume", "delta", "sentiment_zs", "takerbuyvolume"]
df.to_csv("target.csv", columns=col, index=False)
df.tail()

,opentime,open,high,low,close,volume,closetime,quoteassetvolume,numberoftrades,takerbuyvolume,takerbuyquotevolume,ignore,delta,price_effect,vol_min,vol_norm,pricePerVol,sentiment_zs
4994,1772668800000,90.81,91.34,90.67,90.72,687281.37,1772672399999,6.254745e+07,84520,360499.26,3.280897e+07,0,-0.000991,0.2350,460051.91,1.493921,0.157304,-0.155662
4995,1772672400000,90.73,91.38,89.70,89.92,1581302.96,1772675999999,1.427782e+08,137564,723575.37,6.533407e+07,0,-0.008928,1.0275,460051.91,3.437227,0.298933,1.289958
4996,1772676000000,89.92,90.64,89.89,90.13,767492.22,1772679599999,6.927351e+07,82298,418703.29,3.779017e+07,0,0.002335,0.3450,460051.91,1.668273,0.206801,0.330119
4997,1772679600000,90.14,90.49,89.55,89.98,714786.16,1772683199999,6.429551e+07,74548,364429.08,3.279243e+07,0,-0.001775,0.3550,460051.91,1.553708,0.228486,0.541116
4998,1772683200000,89.98,90.69,89.83,90.35,742257.52,1772686799999,6.698523e+07,74050,411607.18,3.715062e+07,0,0.004112,0.4925,460051.91,1.613421,0.305252,1.313086
